# Fraud Detection Pipeline with Kale

This notebook implements a complete fraud detection ML pipeline using **Kubeflow Kale**. Each code cell is annotated with a Kale tag that defines its role in the pipeline.

```mermaid
graph LR
    A[get_data] --> B[preprocess]
    B --> C[train]
    C --> D[evaluate]
    D --> E[upload]

    style A fill:#4394e5,stroke:#2a6cb5,color:#fff
    style B fill:#4394e5,stroke:#2a6cb5,color:#fff
    style C fill:#e5a443,stroke:#b5832a,color:#fff
    style D fill:#3fba73,stroke:#2a8a53,color:#fff
    style E fill:#7b61ff,stroke:#5a45c9,color:#fff
```

**Pipeline steps:**
1. **get_data** — Download training, validation, and test datasets
2. **preprocess** — Scale features and compute class weights
3. **train** — Build and train a Keras DNN
4. **evaluate** — Export to ONNX, run inference, compute metrics
5. **upload** — Push the model to S3 storage

**How to use:**
- **Interactive development:** Run cells top-to-bottom as a normal notebook
- **As a pipeline:** Open the Kale side panel, click "Compile and Run"

Cells tagged `skip` (like this one) are excluded from the pipeline.

In [ ]:
import os
import time
import urllib.request
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
warnings.filterwarnings('ignore')

import matplotlib
import numpy as np
import pandas as pd
from keras import Input
from keras.models import Sequential
from keras.layers import Dense, Dropout, BatchNormalization, Activation
from sklearn.preprocessing import StandardScaler
from sklearn.utils import class_weight
from sklearn.metrics import precision_score, recall_score, confusion_matrix, ConfusionMatrixDisplay
import tf2onnx
import tensorflow as tf
import onnx
import onnxruntime as rt
import pickle
from pathlib import Path
import boto3
import botocore

In [ ]:
def upload_directory_to_s3(local_directory, s3_prefix, bucket):
    """Upload all files in a local directory to an S3 bucket under the given prefix."""
    num_files = 0
    for root, dirs, files in os.walk(local_directory):
        for filename in files:
            file_path = os.path.join(root, filename)
            relative_path = os.path.relpath(file_path, local_directory)
            s3_key = os.path.join(s3_prefix, relative_path)
            print(f"{file_path} -> {s3_key}")
            bucket.upload_file(file_path, s3_key)
            num_files += 1
    return num_files

In [ ]:
epochs = 2
threshold = 0.95
s3_model_key = "models/fraud/1/model.onnx"

In [ ]:
base_url = "https://raw.githubusercontent.com/rh-aiservices-bu/fraud-detection/main/data"
data_dir = "/tmp/data"
os.makedirs(data_dir, exist_ok=True)

print("Downloading datasets...")
urllib.request.urlretrieve(f"{base_url}/train.csv", f"{data_dir}/train.csv")
print("  train.csv downloaded")
urllib.request.urlretrieve(f"{base_url}/validate.csv", f"{data_dir}/validate.csv")
print("  validate.csv downloaded")
urllib.request.urlretrieve(f"{base_url}/test.csv", f"{data_dir}/test.csv")
print("  test.csv downloaded")

df_train = pd.read_csv(f"{data_dir}/train.csv")
df_val = pd.read_csv(f"{data_dir}/validate.csv")
df_test = pd.read_csv(f"{data_dir}/test.csv")

print(f"Train: {len(df_train)} rows, Validate: {len(df_val)} rows, Test: {len(df_test)} rows")

In [ ]:
feature_indexes = [
    1,  # distance_from_last_transaction
    2,  # ratio_to_median_purchase_price
    4,  # used_chip
    5,  # used_pin_number
    6,  # online_order
]
label_indexes = [
    7   # fraud
]

X_train = df_train.iloc[:, feature_indexes].values
y_train = df_train.iloc[:, label_indexes].values

X_val = df_val.iloc[:, feature_indexes].values
y_val = df_val.iloc[:, label_indexes].values

X_test = df_test.iloc[:, feature_indexes].values
y_test = df_test.iloc[:, label_indexes].values

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

artifact_dir = "/tmp/artifact"
Path(artifact_dir).mkdir(parents=True, exist_ok=True)
with open(f"{artifact_dir}/scaler.pkl", "wb") as handle:
    pickle.dump(scaler, handle)

class_weights = class_weight.compute_class_weight(
    'balanced', classes=np.unique(y_train), y=y_train.ravel()
)
class_weights = {i: class_weights[i] for i in range(len(class_weights))}

print(f"Features scaled. Class weights: {class_weights}")

In [ ]:
model = Sequential()
model.add(Input(shape=(len(feature_indexes),)))
model.add(Dense(32, activation='relu'))
model.add(Dropout(0.2))
model.add(Dense(32))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(32))
model.add(BatchNormalization())
model.add(Activation('relu'))
model.add(Dropout(0.2))
model.add(Dense(1, activation='sigmoid'))

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.summary()

start = time.time()
history = model.fit(
    X_train, y_train,
    epochs=epochs,
    validation_data=(X_val, y_val),
    verbose=True,
    class_weight=class_weights
)
elapsed = time.time() - start
print(f"Training complete. Took {elapsed:.1f} seconds")

In [ ]:
if not hasattr(np, 'cast'):
    class _NpCast:
        def __getitem__(self, dtype):
            return lambda x: np.asarray(x, dtype=dtype)
    np.cast = _NpCast()

@tf.function(input_signature=[tf.TensorSpec([None, X_train.shape[1]], tf.float32, name='dense_input')])
def model_fn(x):
    return model(x)

model_proto, _ = tf2onnx.convert.from_function(
    model_fn,
    input_signature=[tf.TensorSpec([None, X_train.shape[1]], tf.float32, name='dense_input')]
)

model_dir = "/tmp/models/fraud/1"
os.makedirs(model_dir, exist_ok=True)
model_path = f"{model_dir}/model.onnx"
onnx.save(model_proto, model_path)
print(f"Model saved to {model_path}")

sess = rt.InferenceSession(model_path, providers=rt.get_available_providers())
input_name = sess.get_inputs()[0].name
output_name = sess.get_outputs()[0].name
y_pred_temp = sess.run([output_name], {input_name: X_test.astype(np.float32)})
y_pred_temp = np.asarray(np.squeeze(y_pred_temp[0]))
y_pred = np.where(y_pred_temp > threshold, 1, 0)

y_test_arr = y_test.squeeze()
correct = np.equal(y_pred, y_test_arr).sum().item()
accuracy = (correct / len(y_pred)) * 100
precision = precision_score(y_test_arr, np.round(y_pred))
recall = recall_score(y_test_arr, np.round(y_pred))

print(f"Accuracy: {accuracy:.1f}%, Precision: {precision:.4f}, Recall: {recall:.4f}")

c_matrix = confusion_matrix(y_test_arr, y_pred)
ConfusionMatrixDisplay(c_matrix).plot()

In [ ]:
aws_access_key_id = os.environ.get('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.environ.get('AWS_SECRET_ACCESS_KEY')
endpoint_url = os.environ.get('AWS_S3_ENDPOINT')
region_name = os.environ.get('AWS_DEFAULT_REGION')
bucket_name = os.environ.get('AWS_S3_BUCKET')

if not all([aws_access_key_id, aws_secret_access_key, endpoint_url, region_name, bucket_name]):
    raise ValueError("One or more S3 connection variables are empty. "
                     "Please check your data connection configuration.")

session = boto3.session.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key
)
s3_resource = session.resource(
    's3',
    config=botocore.client.Config(signature_version='s3v4'),
    endpoint_url=endpoint_url,
    region_name=region_name
)
bucket = s3_resource.Bucket(bucket_name)

print(f"Uploading model to s3://{bucket_name}/{s3_model_key}")
bucket.upload_file(model_path, s3_model_key)
print("Upload complete.")

In [ ]:
print(accuracy)
print(precision)
print(recall)

## Example: Is Sally's transaction fraudulent?

The following cell is tagged `skip` and runs only in interactive mode. It uses the trained model to predict whether a sample transaction is fraudulent.

Fields: distance_from_last_transaction, ratio_to_median_price, used_chip, used_pin_number, online_order

In [ ]:
sally_transaction_details = [
    [0.3111400080477545,
     1.9459399775518593,
     1.0,
     0.0,
     0.0]
]

prediction = sess.run(
    [output_name],
    {input_name: scaler.transform(sally_transaction_details).astype(np.float32)}
)

print("Is Sally's transaction predicted to be fraudulent? (true = YES, false = NO)")
print(np.squeeze(prediction) > threshold)
print(f"Fraud likelihood: {100 * np.squeeze(prediction):.5f}%")